In [3]:
import os
from pathlib import Path
import sys


# 프로젝트 루트 경로를 찾아 src 폴더를 sys.path에 추가합니다.
def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

os.environ.setdefault("LANGSMITH_TRACING", "false")

'true'

In [4]:
# =========================
# 1. 기본 설정
# =========================

import os
import sys
from pathlib import Path

def find_project_root() -> Path:
    cur = Path.cwd()
    for parent in [cur] + list(cur.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return cur

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")

if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

os.environ.setdefault("LANGSMITH_TRACING", "false")

'true'

In [6]:
# =========================
# 2. PDF 로드
# =========================

from langchain_community.document_loaders import PyPDFLoader

file_path = str(PROJECT_ROOT / "data/raw/pdf/welfare/2026_hope_ladder_selected.pdf")

docs = PyPDFLoader(file_path=file_path).load()

print(f"페이지 수: {len(docs)}")

페이지 수: 49


In [7]:
# =========================
# 3. 문서 split
# =========================
# 기존에는 페이지 단위로 바로 벡터화했는데,
# RAG 성능을 위해 chunk 단위로 나누는 게 좋아.

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
)

splits = text_splitter.split_documents(docs)

print(f"chunk 수: {len(splits)}")

chunk 수: 62


In [8]:
# =========================
# 4. 임베딩 + FAISS 벡터스토어 생성
# =========================

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(splits, embeddings)

vectorstore.save_local("hope_rag_index")

print(f"완료: {vectorstore.index.ntotal}개 벡터 저장")

완료: 62개 벡터 저장


In [9]:
# =========================
# 5. 벡터스토어 불러오기
# =========================

vectorstore = FAISS.load_local(
    "hope_rag_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [10]:
# =========================
# 6. Retriever 설정
# =========================

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [33]:
# =========================
# 7. LLM + 프롬프트 설정
# =========================

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

hope_prompt = ChatPromptTemplate.from_template("""
너는 정부 지원 정책을 설명하는 코치야.

반드시 context 기반으로만 답변해.
질문과 직접 관련 있는 정책만 최대 2개까지 설명해.

설명은 자연스럽고 간결한 문장으로 작성해.
"첫 번째", "두 번째" 같은 표현은 사용하지 마.

[context]
{context}

[질문]
{question}

[답변]
""")

In [34]:
# =========================
# 8. 문서 포맷팅 함수
# =========================

def format_docs(docs):
    formatted = []

    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get("page", "unknown")
        content = doc.page_content.strip()

        formatted.append(
            f"[문서 {i} / page {page}]\n{content}"
        )

    return "\n\n".join(formatted)

In [35]:
# =========================
# 9. RAG 답변 함수
# =========================

def ask_hope_rag(question: str):
    docs = retriever.invoke(question)
    context = format_docs(docs)

    chain = hope_prompt | llm

    response = chain.invoke({
        "context": context,
        "question": question
    })

    return {
        "question": question,
        "answer": response.content,
        "source_documents": docs,
        "contexts": [doc.page_content for doc in docs],
    }

In [36]:
# =========================
# 10. 테스트 실행
# =========================

result = ask_hope_rag("임산부 지원 정책 뭐 있어?")

print(result["answer"])

임산부를 위한 지원 정책으로는 맘편한 임신 원스톱 서비스가 있습니다. 이 서비스는 임신 후 받을 수 있는 다양한 지원을 통합적으로 안내하고 신청할 수 있도록 도와줍니다. 엽산제와 철분제 지원, 임신·출산 진료비 지원 등 여러 혜택이 포함되어 있습니다.

또한, 영양플러스 프로그램도 있습니다. 이 프로그램은 임신부와 출산 및 수유부, 5세 이하 영유아를 대상으로 하며, 영양 교육과 상담, 보충식품 패키지를 제공하여 영양 위험 요인을 가진 가구를 지원합니다.


In [37]:
# =========================
# 11. 검색된 근거 문서 확인
# =========================

for i, doc in enumerate(result["source_documents"], 1):
    print(f"\n[{i}] page: {doc.metadata.get('page')}")
    print(doc.page_content[:500])
    print("-" * 80)


[1] page: 20
모두의	정책	K-희망사다리	2026 121
1588-2188
정부24
맘편한 임신 원스톱 서비스
지원대상 	 •	 신청일	기준	임산부
핵심내용 	 •	 임신	후	받을	수	있는	각종	임신지원	서비스를	한	번에	안내받고	통합	신청하는	
서비스
	 •전국	공통	서비스
	 •지자체	서비스
		 ※		지방자치단체별로	제공	내역이	다르므로	관할	읍·면·동	행정복지센터에	확인	
	 •	서비스별	처리	기관에서	문자	또는	유선으로	개별	안내	및	서비스	제공
		 ※		택배	신청	시	엽산·철분제는	건강기능식품	제공,	택배요금은	임산부	부담
	 	 ※	에너지바우처,	산모·신생아	건강관리	지원	신청	시	해당	구비서류가	추가로	필요함
이용방법 	 •온라인 	신청:	정부24(plus.gov.kr)	
	 •방문	신청:	임산부	주소지	읍·면·동	행정복지센터	또는	보건소
문의처	 • 	정부24(☎1588-2188),	정부민원안내(☎110),	행정안전부(☎02-2100-
3399)
구분 서비스
일반서비스
엽산제	지
--------------------------------------------------------------------------------

[2] page: 18
모두의	정책	K-희망사다리	2026 119
월 1회 이상
임산부·영유아	대상	
영양지원	서비스
129
보건복지상담센터
임산부 및 영유아 
영양플러스
지원대상 	 •	 임신부,	출산	및	수유부,	5세	이하	영·유아
	 •	가구	규모별	기준	중위소득	80%	이하	
	 •	영양위험요인	보유자(빈혈,	저체중,	성장부진,	영양섭취상태	불량	등	한	가지	
이상	영양위험	보유)
핵심내용 	
이용방법 	 •	 방문	신청:	거주	지역	보건소
문의처	 •거주	지역	보건소,	보건복지상담센터(☎129)
구분    내용
영양교육 및 상담
	최소	월	1회	이상	실시(집단교육,	소그룹	교육,	가정방문,	
1:1	상담	등)
보충식품 패키지 대상	구분별	여섯	가지	식품	패키지	중	해당	패키지	제공
패키지 구분
①	영아(

In [38]:
question = "임산부 지원 정책 뭐 있어?"

ground_truth = """
맘편한 임신 원스톱 서비스가 있으며,
임산부는 엽산제, 철분제, 임신·출산 진료비 지원 등을 받을 수 있다.
영양플러스는 임산부, 출산 및 수유부, 5세 이하 영유아 중 조건을 충족하는 대상에게 영양교육과 보충식품 패키지를 제공한다.
"""

result = ask_hope_rag(question)

eval_row = {
    "question": question,
    "answer": result["answer"],
    "contexts": [c[:200] for c in result["contexts"][:2]],
    "ground_truth": ground_truth
}

eval_row

{'question': '임산부 지원 정책 뭐 있어?',
 'answer': '임산부를 위한 지원 정책으로는 맘편한 임신 원스톱 서비스가 있습니다. 이 서비스는 임신 후 받을 수 있는 다양한 지원을 통합적으로 안내하고 신청할 수 있도록 도와줍니다. 엽산제와 철분제 지원, 임신·출산 진료비 지원 등 여러 혜택이 포함되어 있습니다.\n\n또한, 영양플러스 프로그램도 있습니다. 이 프로그램은 임신부와 출산 및 수유부, 5세 이하 영유아를 대상으로 하며, 영양 교육과 상담, 보충식품 패키지를 제공하여 영양 위험 요인을 가진 가구를 지원합니다.',
 'contexts': ['모두의\t정책\tK-희망사다리\t2026 121\n1588-2188\n정부24\n맘편한 임신 원스톱 서비스\n지원대상 \t •\t 신청일\t기준\t임산부\n핵심내용 \t •\t 임신\t후\t받을\t수\t있는\t각종\t임신지원\t서비스를\t한\t번에\t안내받고\t통합\t신청하는\t\n서비스\n\t •전국\t공통\t서비스\n\t •지자체\t서비스\n\t\t ※\t\t지방자치단체별로\t제공\t내역이\t다르므로\t관할\t읍·면·동\t행정복지센터',
  '모두의\t정책\tK-희망사다리\t2026 119\n월 1회 이상\n임산부·영유아\t대상\t\n영양지원\t서비스\n129\n보건복지상담센터\n임산부 및 영유아 \n영양플러스\n지원대상 \t •\t 임신부,\t출산\t및\t수유부,\t5세\t이하\t영·유아\n\t •\t가구\t규모별\t기준\t중위소득\t80%\t이하\t\n\t •\t영양위험요인\t보유자(빈혈,\t저체중,\t성장부진,\t영양섭취상태\t불량\t등\t한\t가지\t\n이상\t영양위'],
 'ground_truth': '\n맘편한 임신 원스톱 서비스가 있으며,\n임산부는 엽산제, 철분제, 임신·출산 진료비 지원 등을 받을 수 있다.\n영양플러스는 임산부, 출산 및 수유부, 5세 이하 영유아 중 조건을 충족하는 대상에게 영양교육과 보충식품 패키지를 제공한다.\n'}